# BGF test-set inference (temperature-scaled)

Standalone inference script. For each similarity split, loads that split's model checkpoint,
scores its test CSV at a manually-designated temperature, and writes a per-sequence
`predicted_label` + `confidence` CSV. Does not fit temperatures — fill in `SPLITS` with
values you've already chosen (e.g. from a prior calibration sweep).

In [ ]:
!pip install evaluate prettytable peft
!pip install -U torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 42.7 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import sys
# point this at whatever folder contains train.py
sys.path.insert(0, "/content/drive/MyDrive/bgf_v2")
import os
import re
import json
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, DataCollatorWithPadding
from safetensors.torch import load_file

# pull in the custom classes
from train import ESM2LoRAForSequenceClassification, ESMConfig

Mounted at /content/drive


In [ ]:
class TokenizedDataset(Dataset):
    def __init__(self, csv_file, tokenizer, label_mapper):
        self.data = pd.read_csv(csv_file)
        self.tokenizer = tokenizer
        self.label_mapper = label_mapper

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        sequence = self.data.iloc[idx]['sequence']
        label = self.label_mapper[self.data.iloc[idx]['cycle']]
        inputs = self.tokenizer(sequence)
        return {
            'input_ids':      inputs['input_ids'],
            'attention_mask': inputs['attention_mask'],
            'labels':         label,
        }

In [ ]:
def remap_layernorm_keys(state_dict):
    new_state = {}
    for k, v in state_dict.items():
        new_k = (k.replace(".LayerNorm.gamma", ".LayerNorm.weight")
                  .replace(".LayerNorm.beta", ".LayerNorm.bias"))
        new_state[new_k] = v
    return new_state

In [ ]:
def run_inference_for_split(split_cfg, tokenizer, device):
    """Load one split's checkpoint, score its test CSV at a fixed temperature,
    and write a top-1 + confidence prediction per sequence."""
    name           = split_cfg["name"]
    model_path     = split_cfg["model_path"]
    test_csv       = split_cfg["test_csv"]
    label_map_path = split_cfg["label_map"]
    temperature    = split_cfg["temperature"]

    with open(label_map_path, "r") as f:
        label_mapper = json.load(f)
    id_to_label = {v: k for k, v in label_mapper.items()}

    config = ESMConfig.from_pretrained(model_path)
    state = load_file(os.path.join(model_path, "model.safetensors"))
    state = remap_layernorm_keys(state)   # <-- new line

    n_keep = max(int(m.group(1)) for k in state
                 if (m := re.search(r"encoder\.layer\.(\d+)\.", k))) + 1

    model = ESM2LoRAForSequenceClassification(config)
    base = model.get_submodule("backbone.base_model.model")
    base.encoder.layer = nn.ModuleList(list(base.encoder.layer)[:n_keep])
    base.pooler = None

    inc = model.load_state_dict(state, strict=False)
    assert not inc.missing_keys and not inc.unexpected_keys, inc
    print(f"[{name}] clean load: {n_keep} layers, no pooler")
    model.layer_idx = 15
    model.to(device)
    model.eval()

    test_dataset = TokenizedDataset(test_csv, tokenizer, label_mapper)
    sequences = test_dataset.data["sequence"].tolist()
    collator = DataCollatorWithPadding(tokenizer=tokenizer, padding="longest", return_tensors="pt")
    test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False, collate_fn=collator)

    all_true, all_pred, all_conf = [], [], []
    with torch.no_grad(), torch.autocast(device_type="cuda", dtype=torch.bfloat16):
        for batch in tqdm(test_loader, desc=f"Inference [{name}]", unit="batch"):
            input_ids      = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels         = batch["labels"].to(device)
            _, logits = model(input_ids=input_ids, attention_mask=attention_mask)
            scaled = logits.float() / temperature
            probs = F.softmax(scaled, dim=1)
            conf, pred = probs.max(dim=1)
            all_true.append(labels.cpu())
            all_pred.append(pred.cpu())
            all_conf.append(conf.cpu())

    all_true = torch.cat(all_true)
    all_pred = torch.cat(all_pred)
    all_conf = torch.cat(all_conf)

    acc = (all_pred == all_true).float().mean().item()
    print(f"[{name}] Test accuracy @ T={temperature}: {acc:.4f}")

    out_dir = os.path.join(
        "/content/drive/MyDrive/bgf_v2/results/inference", name
    )
    os.makedirs(out_dir, exist_ok=True)

    results_df = pd.DataFrame({
        "sequence":        sequences,
        "true_label":      [id_to_label[i] for i in all_true.tolist()],
        "predicted_label": [id_to_label[i] for i in all_pred.tolist()],
        "confidence":      all_conf.tolist(),
    })
    out_path = os.path.join(out_dir, "test_predictions.csv")
    results_df.to_csv(out_path, index=False)
    print(f"[{name}] Saved predictions to {out_path}")

    return {"name": name, "temperature": temperature, "accuracy": acc, "n": len(results_df)}

In [ ]:
if __name__ == "__main__":
    tokenizer = AutoTokenizer.from_pretrained("facebook/esm2_t6_8M_UR50D")
    device = "cuda:0" if torch.cuda.is_available() else "cpu"
    # One entry per similarity split. `temperature` is the value you've
    # already chosen (e.g. from a prior calibration sweep on that split's
    # validation set) — this script does not fit temperatures, only applies them.
    SPLITS = [
        {
            "name": "sim_60",
            "model_path": "/content/drive/MyDrive/bgf_v2/models/bgf_150M_60_1ep/checkpoint-3621",
            "test_csv": "/content/drive/MyDrive/bgf_v2/data/test/bgf_test_60.csv",
            "label_map": "/content/drive/MyDrive/bgf_v2/data/cyc_id_60_label_map.json",
            "temperature": 0.8916,
    }]

    summary = []
    for split_cfg in SPLITS:
        result = run_inference_for_split(split_cfg, tokenizer, device)
        summary.append(result)

    print("\n=== Inference summary across splits ===")
    print(f"{'split':>10} {'T':>8} {'acc':>8} {'n':>6}")
    for r in summary:
        print(f"{r['name']:>10} {r['temperature']:>8.3f} {r['accuracy']:>8.4f} {r['n']:>6}")

config.json:   0%|          | 0.00/775 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/95.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/93.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/779 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/486 [00:00<?, ?it/s]

[transformers] EsmModel LOAD REPORT from: facebook/esm2_t30_150M_UR50D
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[sim_60] clean load: 15 layers, no pooler


Inference [sim_60]: 100%|██████████| 9114/9114 [13:52<00:00, 10.94batch/s]


[sim_60] Test accuracy @ T=0.8916: 0.9440
[sim_60] Saved predictions to /content/drive/MyDrive/bgf_v2/results/inference/sim_60/test_predictions.csv

=== Inference summary across splits ===
     split        T      acc      n
    sim_60    0.892   0.9440 145823
